# 诗歌生成

# 数据处理

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r', encoding='utf-8') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [2]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        self.rnncell = tf.keras.layers.LSTMCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, inp_ids):
        x = self.embed_layer(inp_ids)
        x = self.rnn_layer(x)
        
        logits = self.dense(x)
        return logits
    
    def get_next_token(self, x, state):
        inp_emb = self.embed_layer(x) 
        h, state = self.rnncell(inp_emb, state) 
        logits = self.dense(h) 
        
        out = tf.random.categorical(logits, num_samples=1)
        out = tf.reshape(out, [-1])
        out = tf.cast(out, tf.int32) 
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [3]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [4]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    with tf.GradientTape() as tape:
        # 计算模型输出
        logits = model(x)
        # 调用已经给出的 compute_loss
        loss = compute_loss(logits, y, seqlen)
        
    # 计算梯度并更新参数
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [5]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.820493
epoch 1 : loss 6.540348
epoch 2 : loss 6.456056
epoch 3 : loss 6.144809
epoch 4 : loss 6.012541
epoch 5 : loss 5.868507
epoch 6 : loss 5.7713976
epoch 7 : loss 5.667812
epoch 8 : loss 5.551825
epoch 9 : loss 5.510132


# 生成过程

In [8]:
def gen_sentence(start_word):
    # 1. 初始化 LSTM 状态
    state = [tf.random.normal(shape=(1, 128), stddev=0.5), tf.random.normal(shape=(1, 128), stddev=0.5)]
    
    # 2. 先喂入开始符 'bos'，让模型进入准备写诗的状态，并更新 state
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    _, state = model.get_next_token(cur_token, state)
    
    # 3. 强制把我们要的“开头字”设为当前的 token
    start_word_id = word2id.get(start_word, word2id['UNK'])
    cur_token = tf.constant([start_word_id], dtype=tf.int32)
    
    # 将开头的字先装进我们的收集篮里
    collect = [start_word_id]
    
    # 4. 循环生成后续的字（比如再生成最多 50 个字）
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        token_id = cur_token.numpy()[0]
        
        # 遇到结束符 eos，提前结束生成
        if token_id == word2id.get('eos'):
            break
            
        collect.append(token_id)
        
    # 5. 把 id 转换回汉字，并过滤掉不需要显示的特殊字符
    result = [id2word[t] for t in collect if id2word[t] not in ['PAD', 'UNK', 'bos']]
    return ''.join(result)

# ======== 执行题目要求的输出 ========
print("=========== 生成结果 ===========")
start_words = ['日', '红', '山', '夜', '湖', '海', '月']

for word in start_words:
    poem = gen_sentence(word)
    print(f"以【{word}】开头：\n{poem}\n")

=========== 生成结果 ===========
以【日】开头：
日江色一画，秦起中雪满肱。龟游合不水，蹑雨疑稀野，更落明年一城。老学秋寻谁咏，僧却行此还要。

以【红】开头：
红茫拂扆萧窥神，一火年何半万声。日随池烟暮出重，白皂桥鸟草色斜。

以【山】开头：
山余来望壑，惜断幽生所。此事群餔住，琴来出一招。飞中浴黄淡，银蕊耒萝竹。映日终心在，西酒畔桥空。垂舟陪

以【夜】开头：
夜不思草院，霜波翻任禽。风荫风雷净，晚花凄青紫。遥语林前战，圆影不人来。

以【湖】开头：
湖白棹漠边朵好，时沥绕潮成十年处。堕月竹枝随口半，倾径萍桂湖月同边。

以【海】开头：
海堂自大告计天，禅怪惠维名勇师。

以【月】开头：
月兴香外三阙疑，香淡映筠石幌清。人梳山下四云骚，肺学纵尔伤千日。况堪难友语上时首饮，岩夫授口狂声许。

